# 06 — Concurrency & Virtual Threads

**What you'll learn:**

- Why concurrency exists and what a thread *is*
- Starting threads with the `Thread` class
- `Runnable` vs `Callable<V>` — the two task shapes
- Why you almost never create raw threads in modern code
- `ExecutorService` — the pool-based way to run tasks
- `Future<V>` for getting a result back
- **Virtual threads** (Java 21) — millions of cheap threads on one JVM
- Platform threads vs virtual threads — when to use which
- `CompletableFuture` — composing asynchronous work
- **Structured concurrency** (preview) — treating concurrent tasks as a block
- The shared-mutable-state problem — why concurrency is hard
- `synchronized` and intrinsic locks
- `ReentrantLock` and `ReadWriteLock`
- Atomic variables — `AtomicInteger`, `AtomicReference`
- Concurrent collections — `ConcurrentHashMap`, `BlockingQueue`, `CopyOnWriteArrayList`
- The `volatile` keyword and the JVM memory model in one paragraph

Concurrency is where most production bugs live. Modern Java has made the easy cases easier — virtual threads in particular change how you should think about concurrent code — but the hard cases (shared mutable state) still demand care. This notebook walks the easy-to-hard ladder.

## What is a thread?

A **thread** is an independent path of execution. One JVM process can have many threads, each running its own sequence of instructions. The operating system multiplexes them across CPU cores.

```text
   Process (one JVM)
   +-------------------------------+
   |  Thread A    Thread B         |
   |  ----->      ----->           |
   |  ----->      ----->           |
   |              ----->           |
   |  Thread C                     |
   |  ----->                       |
   +-------------------------------+
        \        |         /
         \       |        /
       OS scheduler picks which thread
       runs on each CPU core
```

Why bother? Two reasons:

- **Latency** — while one thread waits on the network or disk, others can keep working.
- **Throughput** — the JVM can spread CPU-bound work across multiple cores.

Threads are essential. They are also the most common source of subtle bugs, because two threads accessing the same memory can interleave in surprising ways.

## Raw threads with the `Thread` class

The lowest-level way to start a thread is to construct one with a `Runnable` and call `.start()`. Don't confuse `.start()` (launches a new thread) with `.run()` (executes the body on the current thread — almost never what you want).

In [ ]:
Thread t = new Thread(() -> {
    for (int i = 0; i < 3; i++) {
        System.out.println("worker: " + i);
    }
});

t.start();          // launches a new OS thread
t.join();           // wait for it to finish (throws InterruptedException)
System.out.println("done");

**You should almost never write `new Thread(...)` in real code.** Reasons:

- Creating an OS thread is expensive — typically ~1 MB of stack memory, plus kernel resources.
- There's no built-in way to limit concurrency, so a loop that spawns threads can exhaust the system.
- There's no convenient way to get a result back.
- There's no built-in lifecycle management — you have to handle interruption, errors, and cleanup yourself.

Use an `ExecutorService` instead. We'll get there in a moment.

## `Runnable` vs `Callable<V>`

Both represent "a task to run on some thread", but they differ in what they return:

| Interface | Method | Returns | Throws checked? |
|---|---|---|---|
| `Runnable` | `void run()` | nothing | no |
| `Callable<V>` | `V call()` | a value | yes — can `throw Exception` |

`Runnable` is the fire-and-forget shape. `Callable<V>` is for tasks that compute a result you want back, possibly throwing a checked exception.

In [ ]:
import java.util.concurrent.*;

// Runnable — no result
Runnable greet = () -> System.out.println("hello");

// Callable<V> — returns a value, can throw checked exceptions
Callable<Integer> compute = () -> {
    Thread.sleep(100);          // throws InterruptedException
    return 42;
};

## `ExecutorService` — pooled task execution

An `ExecutorService` is a managed pool of worker threads that you submit tasks to. It owns the threads; you just hand it work. This is the right abstraction for almost every concurrent workload you write.

In [ ]:
import java.util.concurrent.*;

// pool with a fixed number of platform threads
try (ExecutorService pool = Executors.newFixedThreadPool(4)) {

    // submit a Runnable — no result
    pool.submit(() -> System.out.println("ran on " + Thread.currentThread()));

    // submit a Callable<V> — get a Future back
    Future<Integer> future = pool.submit(() -> {
        Thread.sleep(100);
        return 42;
    });

    Integer result = future.get();      // blocks until the task finishes
    System.out.println("got " + result);

}   // pool.close() — waits for in-flight tasks, then shuts down (Java 19+)

`Executors` is a factory of common pool shapes:

- **`newFixedThreadPool(n)`** — exactly *n* threads; queues excess tasks.
- **`newCachedThreadPool()`** — grows as needed, reuses idle threads. Don't use unbounded for hostile workloads.
- **`newSingleThreadExecutor()`** — exactly one worker; tasks run sequentially.
- **`newVirtualThreadPerTaskExecutor()`** — Java 21+. Spawns a fresh **virtual thread** per task.

Always use `try-with-resources` (Java 19+) or call `.shutdown()` explicitly — the pool's threads are non-daemon by default, so leaving one open prevents the JVM from exiting.

## `Future<V>` — a placeholder for a future result

`Future<V>` is the handle you get when you submit a `Callable`. It represents "the result of an async computation, possibly not ready yet". The main methods:

- **`get()`** — block until the result is ready, then return it. Throws `InterruptedException` and `ExecutionException`.
- **`get(timeout, unit)`** — block at most that long.
- **`cancel(boolean mayInterrupt)`** — try to stop the task.
- **`isDone()`** — non-blocking check.

`Future` is fine for "submit and wait" patterns, but it doesn't compose well — there's no way to say "when this finishes, do something else, without blocking a thread". For that you want `CompletableFuture`, which we'll meet shortly.

## Virtual threads — Java 21's big shift

Until Java 21, every Java thread was a **platform thread** — a thin wrapper around an OS thread. OS threads are expensive: ~1 MB of stack, kernel scheduling overhead, around 10,000 of them in flight on a real machine before things fall over.

**Virtual threads** are a JVM-managed alternative. They are lightweight — kilobytes, not megabytes — and the JVM multiplexes many virtual threads onto a small pool of platform threads (called *carriers*). Blocking a virtual thread on I/O *parks* it; the carrier moves on to another virtual thread.

```text
  millions of virtual threads
   v1  v2  v3  v4  v5  v6  v7  ...
    \   |  /    \   |   /
     +--+--+      +--+--+
     |     |      |     |       <- platform threads (carriers)
     +--+--+      +--+--+         small pool, ~CPU count
        |            |
        +--- OS scheduler ---+
```

The practical effect: you can have **millions** of concurrent virtual threads waiting on the network. Code that used to need a non-blocking framework — Netty, Reactor, async/await gymnastics — can just be written as plain, blocking, one-thread-per-request code, and it scales.

In [ ]:
// the modern shape — one virtual thread per task, no thread sharing
try (var exec = Executors.newVirtualThreadPerTaskExecutor()) {
    var futures = java.util.stream.IntStream.range(0, 10_000)
        .mapToObj(i -> exec.submit(() -> {
            Thread.sleep(100);          // would block a platform thread; parks a virtual one
            return i * 2;
        }))
        .toList();

    int sum = 0;
    for (var f : futures) sum += f.get();
    System.out.println("sum = " + sum);
}

// 10,000 concurrent tasks, no thread pool tuning, no async callbacks

**When to use virtual threads:** I/O-bound work — HTTP servers, database calls, network clients. The whole reason they exist.

**When to use platform threads:** CPU-bound work. A virtual thread that never blocks just pins its carrier and offers no benefit over a platform thread. For raw computation, a `newFixedThreadPool` sized to the CPU count is still the right answer.

**What you should not do:** use `synchronized` on hot paths inside virtual threads. The `synchronized` keyword *pins* the virtual thread to its carrier, defeating the lightweight model. Use `ReentrantLock` instead (we'll see it shortly).

## `CompletableFuture` — composable async

`Future` lets you *wait* for a result; `CompletableFuture<T>` lets you *chain* what to do with it. The shape is the same one you'd recognise from Promises in JavaScript or `Future` in Scala.

In [ ]:
import java.util.concurrent.CompletableFuture;

// supplyAsync — run a Supplier on the common ForkJoinPool, get a CompletableFuture
CompletableFuture<Integer> task = CompletableFuture.supplyAsync(() -> {
    sleep(100);
    return 21;
});

// chain — runs on the same pool when the upstream completes
CompletableFuture<Integer> doubled = task.thenApply(n -> n * 2);

// continuation with side effect
doubled.thenAccept(n -> System.out.println("got " + n));

// blocking wait — usually a sign you should chain instead
int value = doubled.join();         // 42

The three main combinators:

- **`thenApply(Function)`** — transform the result. Like `Stream.map`.
- **`thenCompose(Function)`** — chain another async step that itself returns a `CompletableFuture`. Like `Stream.flatMap` — flattens nested futures.
- **`thenCombine(other, BiFunction)`** — wait for two futures and merge their results.

Plus error handling: `.exceptionally(Throwable -> T)` to recover, `.handle(BiFunction)` to handle both outcomes.

In [ ]:
CompletableFuture<String> a = CompletableFuture.supplyAsync(() -> "hello");
CompletableFuture<String> b = CompletableFuture.supplyAsync(() -> "world");

// combine two independent futures
CompletableFuture<String> joined = a.thenCombine(b, (x, y) -> x + ", " + y);

// wait for many — allOf returns CompletableFuture<Void>
CompletableFuture<Void> all = CompletableFuture.allOf(a, b);
all.join();

// recover from failure
CompletableFuture<Integer> safe = CompletableFuture
    .supplyAsync(() -> { throw new RuntimeException("boom"); })
    .<Integer>handle((value, err) -> err != null ? -1 : value);

## Structured concurrency

`CompletableFuture` is powerful but easy to misuse — orphaned tasks, leaked threads, error handling that gets lost. **Structured concurrency** (preview in Java 21, finalising soon) treats a group of concurrent tasks as a single block: they share a lifetime, errors propagate up, and the block can't exit until every child is done or cancelled.

The shape echoes try-with-resources:

In [ ]:
import java.util.concurrent.StructuredTaskScope;

// pseudo-API — the exact class name has been evolving; this is the Java 21 preview
try (var scope = new StructuredTaskScope.ShutdownOnFailure()) {
    var userTask  = scope.fork(() -> fetchUser(id));
    var orderTask = scope.fork(() -> fetchOrders(id));

    scope.join();                  // wait for both
    scope.throwIfFailed();         // if either threw, propagate

    return new Profile(userTask.get(), orderTask.get());
}
// when the block exits, any running fork is cancelled automatically

The structured model gives you three guarantees the unstructured one didn't: tasks **never outlive** the block that started them, errors **never get swallowed**, and cancellation **propagates** automatically. Treat it as the future direction of concurrent Java — once it's out of preview, prefer it over hand-rolled `CompletableFuture` graphs.

## The shared-mutable-state problem

When two threads touch the same field without coordination, you get **data races** — interleavings that produce wrong results, sometimes deterministically, sometimes once in a million runs.

In [ ]:
class Counter {
    int value = 0;
    void increment() { value++; }   // looks atomic; isn't
}

var c = new Counter();
try (var pool = Executors.newFixedThreadPool(8)) {
    for (int i = 0; i < 10_000; i++) {
        pool.submit(c::increment);
    }
}
System.out.println(c.value);   // usually NOT 10000 — lost updates from interleavings

`value++` looks like one operation but compiles to three: read, add one, write. Two threads can both read the same value, both write back the same incremented number, and one update is silently lost. This is the canonical race condition.

There are three ways to fix it, in increasing sophistication: **synchronization**, **explicit locks**, and **atomic variables**.

## `synchronized` — intrinsic locks

Every Java object has an **intrinsic lock**. The `synchronized` keyword takes that lock for the duration of a block or method. Only one thread at a time can hold a given lock; others block until it's released.

In [ ]:
class SafeCounter {
    private int value = 0;

    // synchronized method — locks `this`
    public synchronized void increment() {
        value++;
    }

    public synchronized int get() {
        return value;
    }
}

// or with a block — locks a specific object
class AnotherSafe {
    private final Object lock = new Object();
    private int value = 0;

    public void increment() {
        synchronized (lock) {
            value++;
        }
    }
}

`synchronized` is simple and built into the language, but it has limits: no timeout, no try-lock, no fairness control. And — critical for virtual threads — `synchronized` *pins* a virtual thread to its carrier, so it should be avoided on the hot path of virtual-thread code.

## `ReentrantLock` — explicit locks

For everything `synchronized` can't do, reach for `ReentrantLock` from `java.util.concurrent.locks`. Same semantics — one holder at a time — plus `tryLock`, timeouts, interruptible acquisition, and (critically) it does **not** pin virtual threads.

In [ ]:
import java.util.concurrent.locks.*;

class BetterCounter {
    private final ReentrantLock lock = new ReentrantLock();
    private int value = 0;

    public void increment() {
        lock.lock();
        try {
            value++;
        } finally {
            lock.unlock();          // ALWAYS in finally
        }
    }

    public boolean tryIncrement() {
        if (!lock.tryLock()) return false;
        try {
            value++;
            return true;
        } finally {
            lock.unlock();
        }
    }
}

There's also **`ReadWriteLock`** — separate read and write locks, allowing multiple concurrent readers but only one writer. Use it for read-heavy data structures where reads dominate writes.

## Atomic variables — lock-free for simple state

For a single value updated concurrently, the `java.util.concurrent.atomic` package is faster than any lock. Atomic classes use CPU-level compare-and-swap instructions and never block.

In [ ]:
import java.util.concurrent.atomic.*;

var counter = new AtomicInteger(0);
counter.incrementAndGet();          // atomic ++
counter.addAndGet(5);
counter.compareAndSet(6, 100);      // set to 100 only if currently 6

var ref = new AtomicReference<String>("initial");
ref.set("updated");
ref.compareAndSet("updated", "final");

// updateAndGet — apply a function atomically
var c = new AtomicInteger(0);
c.updateAndGet(n -> n + 1);
c.accumulateAndGet(10, Integer::sum);

## Concurrent collections

The plain `HashMap` and `ArrayList` are not thread-safe — concurrent modification gives undefined behaviour. The `java.util.concurrent` package supplies thread-safe replacements designed for high contention.

| Class | Replaces | Notes |
|---|---|---|
| **`ConcurrentHashMap`** | `HashMap` | Fine-grained locking; great for high-concurrency reads/writes |
| **`CopyOnWriteArrayList`** | `ArrayList` | Read-heavy; every write copies the array |
| **`BlockingQueue`** (e.g. `LinkedBlockingQueue`) | — | Producer-consumer hand-off |
| **`ConcurrentLinkedQueue`** | — | Lock-free FIFO queue |

In [ ]:
import java.util.concurrent.*;

var counters = new ConcurrentHashMap<String, AtomicInteger>();

// the classic concurrent counter pattern
counters.computeIfAbsent("hits", k -> new AtomicInteger()).incrementAndGet();

// BlockingQueue — producer/consumer hand-off
BlockingQueue<String> queue = new LinkedBlockingQueue<>(100);

// producer
new Thread(() -> {
    try { queue.put("event"); }    // blocks if full
    catch (InterruptedException e) { Thread.currentThread().interrupt(); }
}).start();

// consumer
new Thread(() -> {
    try {
        String event = queue.take();    // blocks if empty
        System.out.println(event);
    } catch (InterruptedException e) { Thread.currentThread().interrupt(); }
}).start();

## `volatile` and the JVM memory model — in one paragraph

Without `volatile`, the compiler and CPU can cache field reads in registers or per-core caches indefinitely. Two threads can disagree about the value of a plain field forever. The **`volatile`** keyword on a field forces every read to come from main memory and every write to flush to main memory. It does *not* make compound operations atomic — `volatile int x; x++;` is still racy because the increment is read-modify-write. Use `volatile` for simple flags (like `volatile boolean shutdown`) and reach for `AtomicInteger`/`AtomicReference` for anything you actually need to update concurrently.

## What's next

You can now write concurrent Java — virtual threads for I/O-bound services, platform threads for CPU work, atomic and concurrent utilities for shared state, and `CompletableFuture` (or structured concurrency, soon) for composing async work. Spring services lean on every primitive in this notebook — every request handler runs on a thread, every database call is a blocking operation that virtual threads make cheap, and every shared cache uses a `ConcurrentHashMap` or similar under the hood.

Notebook 07 turns to the **runtime and the tooling** that wraps everything we've built so far: the **JVM** itself (heap, stack, metaspace, garbage collection at a glance), **reflection** and how it powers Spring's annotation-driven model, **Maven and Gradle** as the build tools every Spring project uses, and **JUnit 5 + Mockito** for testing.